# BTC Data Pull via Massive API
Pulls full BTC/USD daily data and saves to `data/btc_daily.csv`.
Only run once to populate the data file.

In [28]:
import requests
import pandas as pd
import os
from datetime import date
from urllib.parse import quote

API_KEY = '7gjThzOOPPal8jKWBbDl8lgp6ZM1DSYh'
BASE_URL = 'https://api.massive.com'
os.makedirs('data', exist_ok=True)

In [29]:
TICKER = 'X:BTCUSD'
# data starts from 2013-11-01
FROM_DATE = '2013-11-01'
TO_DATE = str(date.today())

ticker_encoded = quote(TICKER, safe='')
url = f'{BASE_URL}/v2/aggs/ticker/{ticker_encoded}/range/1/day/{FROM_DATE}/{TO_DATE}'
params = {
    'adjusted': 'true',
    'sort': 'asc',
    'limit': 50000,
    'apiKey': API_KEY
}

all_results = []

while url:
    resp = requests.get(url, params=params)
    print(f'Status: {resp.status_code}')

    data = resp.json()

    results = data.get('results', [])
    all_results.extend(results)

    url = data.get('next_url')
    params = {}

print(f'\nTotal rows fetched: {len(all_results)}')

Status: 200

Total rows fetched: 4675


In [30]:
df = pd.DataFrame(all_results)

# Rename columns 
df = df.rename(columns={
    't': 'timestamp_ms',
    'o': 'open',
    'h': 'high',
    'l': 'low',
    'c': 'close',
    'v': 'volume',
    'vw': 'vwap',
    'n': 'num_trades'
})

# Convert ms timestamp to date
df['date'] = pd.to_datetime(df['timestamp_ms'], unit='ms').dt.date
df = df[['date', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'num_trades']]
df = df.sort_values('date').reset_index(drop=True)

df.to_csv('data/btc_daily.csv', index=False)
df.head()

,date,open,high,low,close,volume,vwap,num_trades
0,2013-11-01,203.59999,207.96822,202.90000,202.90000,33.980533,205.1009,117
1,2013-11-02,202.68000,204.24000,201.04000,203.51000,7.055556,202.0084,13
2,2013-11-03,204.66200,209.74800,203.47000,209.74800,9.585259,206.7743,120
3,2013-11-04,209.74800,229.97000,209.40010,226.90010,71.175421,219.7584,484
4,2013-11-05,227.39493,250.14053,222.05236,244.95532,25.221364,237.2967,309


Simple Data checks

In [31]:
print('=== Shape ===')
print(df.shape)

print('\n=== Date range ===')
print(f'From : {df["date"].min()}')
print(f'To   : {df["date"].max()}')

print('\n=== Missing values ===')
print(df.isnull().sum())

=== Shape ===
(4675, 8)

=== Date range ===
From : 2013-11-01
To   : 2026-08-29

=== Missing values ===
date          0
open          0
high          0
low           0
close         0
volume        0
vwap          0
num_trades    0
dtype: int64


In [32]:
# summary stats
df.describe()

,open,high,low,close,volume,vwap,num_trades
count,4675.000000,4675.000000,4675.000000,4675.000000,4675.000000,4675.000000,4.675000e+03
mean,27418.868282,28027.242448,26773.768614,27434.769628,32660.671564,27408.651814,3.254682e+05
std,32229.124516,32799.601633,31623.106690,32236.863033,40883.637629,32217.787872,3.257758e+05
min,175.000000,202.172500,0.060000,175.000000,0.000038,199.628400,1.000000e+00
25%,977.095000,1024.445000,910.225000,985.000500,2915.248593,974.579450,2.811600e+04
50%,10377.020000,10713.000000,10051.000000,10389.500000,19155.989567,10372.440900,2.503930e+05
75%,45818.565000,47149.180000,44170.735000,45864.365000,47619.020715,45658.309500,5.025110e+05
max,124765.900000,126296.000000,123115.770000,124720.090000,528732.390968,124887.519800,3.224462e+06


In [33]:
# Check for any missing days in subsetted range
df['date'] = pd.to_datetime(df['date'])
full_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='D')
missing_dates = full_range.difference(df['date'])

print(f'Missing dates: {len(missing_dates)}')

Missing dates: 10


In [34]:
df.tail()

,date,open,high,low,close,volume,vwap,num_trades
4670,2026-08-25,78981.59,81300.0,77832.00,78526.80,15749.198615,79473.7385,1657545
4671,2026-08-26,78509.50,79270.0,77627.00,79008.60,2837.918108,78478.7295,142903
4672,2026-08-27,79026.18,80848.4,78551.99,80275.34,17313.903566,79917.1690,928325
4673,2026-08-28,80275.35,81500.0,76845.71,77839.19,22525.332426,78885.2195,1041980
4674,2026-08-29,77841.80,77968.0,77345.28,77700.19,2760.556529,77640.9277,278086


In [35]:
weekly = df.resample('W', on='date').size().rename('daily_obs')
weeks_with_data = (weekly > 0).sum()
total_daily_obs = len(df)

print(f'Total daily rows: {total_daily_obs}')
print(f'Total weeks: {len(weekly)}')

Total daily rows: 4675
Total weeks: 670
